In [ ]:
from PIL import Image
from kraken.lib import models
from kraken import blla
from kraken import rpred

# Pfade zu deinen Modellen (Beispiele)
SEGMENTATION_MODEL_PATH = 'pfad/zum/segmentation_model.mlmodel'
OCR_MODEL_PATH = 'pfad/zum/fraktur_oder_suetterlin.mlmodel'

# 1. Bild laden
img_path = 'dein_historisches_dokument.jpg'
img = Image.open(img_path)

# --- ZWISCHENSCHRITT 1: Image Preprocessing ---
# Hier kannst du mit OpenCV (cv2) oder scikit-image eingreifen.
# z.B. Denoising, Kontrastverstärkung, Deskewing.
# Wichtig: Konvertiere das Bild danach wieder in ein PIL.Image, da Kraken das erwartet.
# ----------------------------------------------

# 2. Modelle laden
# Dies dauert einen Moment, lade sie im Notebook am besten in einer separaten Zelle.
seg_model = models.load_any(SEGMENTATION_MODEL_PATH)
ocr_model = models.load_any(OCR_MODEL_PATH)

# 3. Segmentierung (Layout Analysis)
# Findet Textzeilen (Baselines) und Regionen
res = blla.segment(img, model=seg_model)

# --- ZWISCHENSCHRITT 2: Layout filtern ---
# 'res' ist ein Dictionary mit 'lines' und 'regions'.
# res['lines'] enthält Koordinaten der Baselines und Boundary Polygons.
# Hier kannst du eingreifen: Zeilen löschen, die zu kurz sind, oder 
# Polygons anpassen, die Artefakte vom Seitenrand enthalten.
# -----------------------------------------

# 4. Texterkennung (OCR/HTR)
# rpred ist ein Generator. Er nimmt das Bild und die gefundenen Zeilen (bounds=res)
pred_it = rpred.rpred(network=ocr_model, im=img, bounds=res)

# 5. Ergebnisse extrahieren
results = []
for record in pred_it:
    # record.prediction enthält den erkannten Text der Zeile
    # record.cuts enthält die Koordinaten der einzelnen Zeichen
    # record.confidences enthält die Konfidenzwerte pro Zeichen
    results.append({
        'text': record.prediction,
        'confidences': record.confidences,
        'polygon': record.line
    })

# Ausgabe der ersten Zeilen
for r in results[:5]:
    print(f"[{sum(r['confidences'])/len(r['confidences']):.2f}] {r['text']}")